In [1]:
print("hello world")

hello world


In [2]:
# Install the required libraries (use %pip for reliable in-notebook installation)
%pip install pytorch-forecasting lightning openpyxl pandas matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# UPDATE: Use the modern lightning.pytorch namespace instead of pytorch_lightning
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE

# 1. Load the data
df = pd.read_excel('Demand of last 2 years.xlsx', sheet_name='demand_training_data')

# 2. Convert raw dates into a continuous integer timeline
# WHY: Deep learning math cannot process "2024-01-01". It needs step 0, step 1, step 2.
df['week_start_date'] = pd.to_datetime(df['week_start_date'], errors='coerce')
date_map = {date: idx for idx, date in enumerate(sorted(df['week_start_date'].unique()))}
df['time_idx'] = df['week_start_date'].map(date_map).astype(int)

# 3. Fill missing blanks 
# WHY: A blank cell in Excel becomes 'NaN' in Python, which breaks the tensor calculations.
df['festival_event'] = df['festival_event'].fillna('None')
df['weather_condition'] = df['weather_condition'].fillna('Unknown')
df['season'] = df['season'].fillna('Unknown')

# 4. Force strict data types
# WHY: TFT requires distinct groups. Categoricals MUST be strings. Metrics MUST be floats.
categorical_features = [
    'location_id', 'region_of_kolkata', 'product_id', 'product_name', 
    'category_name', 'brand', 'unit_size', 'season', 'festival_event', 'weather_condition'
]
for col in categorical_features:
    df[col] = df[col].astype(str)

numeric_features = [
    'units_sold', 'avg_selling_price_rs', 'temperature_c', 'rainfall_mm', 'humidity_pct', 'discount_pct',
    'week_number', 'year', 'promotion_flag', 'holiday_flag'
]
for col in numeric_features:
    df[col] = df[col].fillna(0).astype(float)

print("Data cleaned successfully. Rows:", len(df))
max_encoder_length = 12   # Look back 12 weeks into historical data
max_prediction_length = 4 # Forecast demand 4 weeks into the future

# Calculate the 80% cutoff point in the timeline
max_time_idx = df['time_idx'].max()
training_cutoff = int(max_time_idx * 0.8)

# 1. Define the Training Dataset (First 80% of time)
# WHY: We strictly isolate the first 80% so the model learns patterns without seeing the final 20%
training_dataset = TimeSeriesDataSet(
    df[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="units_sold",
    group_ids=["location_id", "product_id"], # Keeps distinct items at distinct hubs separated
    
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    
    static_categoricals=["location_id", "region_of_kolkata", "product_id", "category_name", "brand", "unit_size"],
    time_varying_known_categoricals=["season", "festival_event"],
    time_varying_known_reals=["time_idx", "week_number", "year", "promotion_flag", "holiday_flag"],
    time_varying_unknown_categoricals=["weather_condition"],
    time_varying_unknown_reals=["units_sold", "avg_selling_price_rs", "discount_pct", "temperature_c", "rainfall_mm", "humidity_pct"],
    
    target_normalizer=GroupNormalizer(
        groups=["location_id", "product_id"], transformation="softplus"
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True, # WHY: Prevents the model from crashing if a product wasn't sold for a week
)

# 2. Define the Validation Dataset (Last 20% of time)
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, 
    df, 
    predict=False, 
    min_prediction_idx=training_cutoff + 1, # WHY: Forces evaluation ONLY on the unseen 20%
    stop_randomization=True
)

# 3. Create DataLoaders to feed batches into the model
train_dataloader = training_dataset.to_dataloader(train=True, batch_size=32, num_workers=0)
val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=64, num_workers=0)

print("Datasets configured successfully.")
# 1. Initialize the TFT Architecture
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.03,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(), 
    reduce_on_plateau_patience=4,
)

# 2. Setup Callbacks using the new namespace
early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=5, verbose=True, mode="min")
checkpoint_callback = ModelCheckpoint(monitor="val_loss", mode="min")

# 3. Run the Trainer
trainer = pl.Trainer(
    max_epochs=2,
    accelerator="auto", 
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback, checkpoint_callback],
)

print("Starting training loop...")
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)


Note: you may need to restart the kernel to use updated packages.


Data cleaned successfully. Rows: 27800


C:\Users\kaifa\ai-multi-agent-supply-chain\.venv\Lib\site-packages\pytorch_forecasting\data\timeseries\_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 325 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__location_id': 'KOL-LOC-001', '__group_id__product_id': 'BEV-010'}, {'__group_id__location_id': 'KOL-LOC-001', '__group_id__product_id': 'DAI-005'}, {'__group_id__location_id': 'KOL-LOC-001', '__group_id__product_id': 'DAI-010'}, {'__group_id__location_id': 'KOL-LOC-001', '__group_id__product_id': 'ICE-003'}, {'__group_id__location_id': 'KOL-LOC-001', '__group_id__product_id': 'ICE-007'}, {'__group_id__location_id': 'KOL-LOC-001', '__group_id__product_id': 'SNK-001'}, {'__group_id__location_id': 'KOL-LOC-001', '__group_id__product_id': 'SNK-004'}, {'__group_id__location_id': 'KOL-LOC-0

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


...

C:\Users\kaifa\ai-multi-agent-supply-chain\.venv\Lib\site-packages\pytorch_forecasting\metrics\base_metrics\_base_metrics.py:906: UserWarning: Loss is not finite. Resetting it to 1e9
  warnings.warn("Loss is not finite. Resetting it to 1e9")


Metric val_loss improved. New best score: 177840.469


Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.016 >= min_delta = 0.0001. New best score: 177840.453


`Trainer.fit` stopped: `max_epochs=2` reached.


In [3]:
# 5. Generate and Inspect Demand Forecasts
# Load the best checkpoint produced by the trainer
best_model_path = trainer.checkpoint_callback.best_model_path
print(f"Loading best model checkpoint: {best_model_path}")
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

# Create a prediction dataset to forecast the future 4 weeks
pred_dataset = TimeSeriesDataSet.from_dataset(training_dataset, df, predict=True, stop_randomization=True)
pred_dataloader = pred_dataset.to_dataloader(train=False, batch_size=64, num_workers=0)

# Generate predictions across all series
predictions = best_tft.predict(pred_dataloader)

print("Forecast generation successful!")
print(f"Total series forecasted: {predictions.shape[0]}")
print(f"Forecast horizon: {predictions.shape[1]} weeks")

# Display sample series forecasts
print("\n--- Sample 4-Week Ahead Forecasts ---")
for i in range(min(5, predictions.shape[0])):
    forecast_vals = [round(float(v), 1) for v in predictions[i]]
    print(f"Series {i + 1} predicted demand (Weeks +1 to +4): {forecast_vals} units")


Loading best model checkpoint: lightning_logs\version_2\checkpoints\epoch=1-step=1502.ckpt
Forecast generation successful!
Total series forecasted: 520
Forecast horizon: 4 weeks

--- Sample 4-Week Ahead Forecasts ---
Series 1 predicted demand (Weeks +1 to +4): [129.9, 116.4, 115.9, 116.2] units
Series 2 predicted demand (Weeks +1 to +4): [113.6, 116.7, 120.2, 112.3] units
Series 3 predicted demand (Weeks +1 to +4): [152.9, 141.4, 123.5, 123.6] units
Series 4 predicted demand (Weeks +1 to +4): [148.7, 115.9, 115.3, 115.5] units
Series 5 predicted demand (Weeks +1 to +4): [72.1, 74.6, 75.1, 81.9] units
